# Assessor evaluation and optimization

Executed offline on 2026-11-15. Real LangGraph, Pydantic, SQLite and MemorySaver. Actual execution needs the verified Linux sandbox. The later optimizer examples use explicitly labeled known execution fixtures; they are not live learner-code execution or OpenAI inference.

In [1]:
from pathlib import Path
import os, tempfile, json
workspace = Path.cwd()
if not (workspace/'src').exists(): workspace=workspace.parent
import sys
sys.path.insert(0,str(workspace))
run_dir = tempfile.TemporaryDirectory(prefix='sprints-notebook-')
os.environ.update(SPRINTS_LLM='offline', SPRINTS_EMBEDDINGS='hashing', SPRINTS_LIVE_AUTHORIZED='0', SPRINTS_CHROMA_DIR=str(Path(run_dir.name)/'chroma'), SPRINTS_DB_PATH=str(Path(run_dir.name)/'learners.db'), ANONYMIZED_TELEMETRY='False', LANGSMITH_TRACING='false', LANGCHAIN_TRACING_V2='false')
from src.auth import MockIdentityProvider
from src.service import OrchestratorService
from src.main import BUGGY, FIXED
from src.tools.code_repl import sandbox_capabilities
print(json.dumps(sandbox_capabilities(),indent=2))

{
  "available": false,
  "backend": "linux-docker",
  "status": "unable_to_verify",
  "reason": "Local Linux Docker daemon unavailable"
}


## Actual safe execution path
The application returns structured Unable to verify when containment is unavailable. No host fallback is permitted and no grade is invented.

In [2]:
identity=MockIdentityProvider()
app=OrchestratorService(identity,warm=False)
session=identity.issue_mock_session('learner-1005')
actual=[]
for submission in (BUGGY,FIXED):
    result=app.ask(session,{'text':submission})
    actual.append(result)
    print(json.dumps(result,indent=2))
if not sandbox_capabilities()['available']:
    assert all(r['status']=='unable_to_verify' and not r.get('assessment') for r in actual)

{
  "status": "unable_to_verify",
  "route": "assessor",
  "final_answer": "Unable to verify: a verified isolated Python sandbox is unavailable (source: python_repl).",
  "turn_count": 1,
  "assessment": {},
  "refinement_count": 0,
  "grader_calls": 0,
  "needs_human_review": false,
  "retrieval_attempts": 0,
  "fallback": {
    "status": "unable_to_verify",
    "source": "python_repl",
    "message": "Unable to verify"
  }
}


{
  "status": "unable_to_verify",
  "route": "assessor",
  "final_answer": "Unable to verify: a verified isolated Python sandbox is unavailable (source: python_repl).",
  "turn_count": 2,
  "assessment": {},
  "refinement_count": 0,
  "grader_calls": 0,
  "needs_human_review": false,
  "retrieval_attempts": 0,
  "fallback": {
    "status": "unable_to_verify",
    "source": "python_repl",
    "message": "Unable to verify"
  }
}


## Optimizer control flow using execution fixtures
These known fixture outcomes exercise feedback refinement and the three-refinement cap through the real graph. The fixture is never presented as measured Python execution. The judge here is deterministic, not an LLM.

In [3]:
from unittest.mock import patch
from tests.test_evaluator_optimizer import execution_fixture, LearnsFromJudge, Inflater
fixture_results=[]
for reasoner in (LearnsFromJudge(),Inflater()):
    with patch('src.agents.assessor_agent.run_with_tests',execution_fixture):
        fixture_app=OrchestratorService(identity,reasoner=reasoner,warm=False)
        result=fixture_app.ask(session,{'text':BUGGY})
    fixture_results.append(result)
    print('MOCK_EXECUTION_FIXTURE',type(reasoner).__name__,json.dumps(result,indent=2))
assert fixture_results[0]['refinement_count']==1
assert fixture_results[1]['refinement_count']==3
assert fixture_results[1]['grader_calls']==4
assert fixture_results[1]['status']=='graded_needs_human_review'

MOCK_EXECUTION_FIXTURE LearnsFromJudge {
  "status": "graded",
  "route": "assessor",
  "final_answer": "Score: 25/100 - FAIL (pass mark 70)\n- Passed 0/6 tests (correctness 0/70).\n- Test failed for input ([80, None, 90],): expected 85.0, got fixture failure.\n- Test failed for input ([],): expected 0.0, got fixture failure.\n- Readability: add a docstring explaining what the code does (-5).",
  "turn_count": 1,
  "assessment": {
    "score": 25,
    "feedback_points": [
      "Passed 0/6 tests (correctness 0/70).",
      "Test failed for input ([80, None, 90],): expected 85.0, got fixture failure.",
      "Test failed for input ([],): expected 0.0, got fixture failure.",
      "Readability: add a docstring explaining what the code does (-5)."
    ],
    "pass_status": "fail"
  },
  "refinement_count": 1,
  "grader_calls": 2,
  "needs_human_review": false,
  "retrieval_attempts": 0
}
MOCK_EXECUTION_FIXTURE Inflater {
  "status": "graded_needs_human_review",
  "route": "assessor",
  "f

In [4]:
with patch('src.agents.assessor_agent.run_with_tests',execution_fixture):
    corrected=OrchestratorService(identity,warm=False).ask(session,{'text':FIXED})
print('MOCK_EXECUTION_FIXTURE corrected',json.dumps(corrected,indent=2))
assert corrected['assessment']['score']==100
assert corrected['status']=='graded'

MOCK_EXECUTION_FIXTURE corrected {
  "status": "graded",
  "route": "assessor",
  "final_answer": "Score: 100/100 - PASS (pass mark 70)\n- Passed 6/6 tests (correctness 70/70).\n- All tests passed and the code is clean and readable. Excellent work.",
  "turn_count": 1,
  "assessment": {
    "score": 100,
    "feedback_points": [
      "Passed 6/6 tests (correctness 70/70).",
      "All tests passed and the code is clean and readable. Excellent work."
    ],
    "pass_status": "pass"
  },
  "refinement_count": 0,
  "grader_calls": 1,
  "needs_human_review": false,
  "retrieval_attempts": 0
}


## Interpretation
The notebook verifies typed grading output, bounded optimization, and safe degradation. Container containment and live OpenAI judgement remain unverified when unavailable. The 80% waiting-time reduction remains a stakeholder target; these timings do not measure business queue waiting.